# 📏 Les Métriques d'Évaluation — Notebook Bootcamp Data Science

## Module Data Science — Machine Learning

> **Prérequis :** cours *Algorithmes de ML* et *Train/Test Split & Validation Croisée*.

Ce notebook accompagne le cours `04-DataScience_Evaluation_Metrics.md`.
Il est **autonome** : il utilise les jeux de données intégrés à scikit-learn, aucun CSV externe requis.

**Plan :**
1. Classification — matrice de confusion, Accuracy, Precision, Recall, F1, ROC/AUC
2. Régression — MAE, MSE, RMSE, RMSLE, R²
3. Clustering — Silhouette, Davies-Bouldin, Inertie (coude), ARI

In [ ]:
!pip install scikit-learn seaborn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
sns.set_theme(style='whitegrid')
print('Environnement prêt ✅')

---
# PARTIE I — Métriques de Classification

On utilise le dataset **Breast Cancer** (diagnostic : tumeur maligne / bénigne).
C'est un problème **binaire** où **rater un cas positif (malignité)** est grave → le **Recall** comptera.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

data = load_breast_cancer()
X, y = data.data, data.target
# NB : dans ce dataset, classe 0 = malin, 1 = bénin. On garde tel quel.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Pipeline (normalisation + modèle) pour éviter toute fuite de données
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
clf.fit(X_train, y_train)

y_pred  = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]   # proba de la classe positive (1)
print('Modèle entraîné ✅  —  répartition test :', np.bincount(y_test))

## 1. La matrice de confusion

Elle croise **prédictions** et **vérité**. Presque toutes les métriques en découlent (VP, VN, FP, FN).

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)
print('Matrice de confusion :')
print(cm)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=data.target_names).plot(ax=ax, cmap='Blues')
ax.set_title('Matrice de confusion')
plt.tight_layout(); plt.show()

## 2 à 4. Accuracy, Precision, Recall, F1

- **Accuracy** = bonnes prédictions / total (⚠️ trompeuse si classes déséquilibrées)
- **Precision** = VP / (VP+FP) → punit les **faux positifs**
- **Recall** = VP / (VP+FN) → punit les **faux négatifs**
- **F1** = moyenne harmonique de Precision & Recall

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, classification_report)

print(f'Accuracy  : {accuracy_score(y_test, y_pred):.3f}')
print(f'Precision : {precision_score(y_test, y_pred):.3f}')
print(f'Recall    : {recall_score(y_test, y_pred):.3f}')
print(f'F1 Score  : {f1_score(y_test, y_pred):.3f}')
print()
print('Rapport complet (par classe) :')
print(classification_report(y_test, y_pred, target_names=data.target_names))

### ⚖️ Le compromis Precision ↔ Recall via le seuil

Par défaut on prédit *positif* si la proba > 0.5. **Baisser le seuil** augmente le Recall (moins de ratés)
mais baisse la Precision (plus de fausses alertes), et inversement.

In [ ]:
for seuil in [0.3, 0.5, 0.7]:
    y_s = (y_proba >= seuil).astype(int)
    p = precision_score(y_test, y_s)
    r = recall_score(y_test, y_s)
    print(f'Seuil {seuil} → Precision={p:.3f} | Recall={r:.3f}')

print('\n👉 Seuil bas = Recall ↑ / Precision ↓  ;  seuil haut = l\'inverse.')

## 5. Courbe ROC et AUC

L'**AUC** résume la qualité du modèle **à tous les seuils** en un seul nombre (0.5 = hasard, 1.0 = parfait).
Elle utilise les **probabilités**, pas les classes dures.

In [ ]:
from sklearn.metrics import roc_auc_score, RocCurveDisplay, roc_curve

auc = roc_auc_score(y_test, y_proba)
print(f'AUC = {auc:.3f}')

fpr, tpr, _ = roc_curve(y_test, y_proba)
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot(fpr, tpr, label=f'Modèle (AUC = {auc:.3f})', lw=2)
ax.plot([0, 1], [0, 1], '--', color='grey', label='Hasard (AUC = 0.5)')
ax.set_xlabel('Taux de Faux Positifs'); ax.set_ylabel('Taux de Vrais Positifs (Recall)')
ax.set_title('Courbe ROC'); ax.legend()
plt.tight_layout(); plt.show()

### 🧩 Compléments : Log Loss & moyennes multiclasses

- **Log Loss** : qualité des probabilités (pénalise une prédiction confiante ET fausse). Plus bas = mieux.
- En **multiclasse**, F1 se moyenne en `macro` (chaque classe à égalité) ou `weighted` (pondéré).

In [ ]:
from sklearn.metrics import log_loss
print(f'Log Loss : {log_loss(y_test, clf.predict_proba(X_test)):.3f}  (plus bas = mieux)')

# Exemple multiclasse rapide (Iris) pour macro vs weighted
from sklearn.datasets import load_iris
Xi, yi = load_iris(return_X_y=True)
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.3, random_state=42, stratify=yi)
clf_i = LogisticRegression(max_iter=500).fit(Xi_tr, yi_tr)
yi_pred = clf_i.predict(Xi_te)
print(f"F1 macro    : {f1_score(yi_te, yi_pred, average='macro'):.3f}")
print(f"F1 weighted : {f1_score(yi_te, yi_pred, average='weighted'):.3f}")

---
# PARTIE II — Métriques de Régression

On utilise le dataset **California Housing** (prédire un prix médian de logement).
On mesure l'**écart** entre valeur prédite et valeur réelle.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor

housing = fetch_california_housing()
Xr, yr = housing.data, housing.target   # yr en centaines de milliers de $
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.2, random_state=42)

reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
reg.fit(Xr_tr, yr_tr)
yr_pred = reg.predict(Xr_te)
print('Modèle de régression entraîné ✅')

## 1 à 3 & 5. MAE, MSE, RMSE, R²

- **MAE** : erreur absolue moyenne (lisible, robuste aux outliers)
- **MSE** : erreur au carré (punit les gros écarts, unité²)
- **RMSE** : √MSE (lisible **et** sévère) — le standard
- **R²** : part de variance expliquée (1 = parfait, 0 = niveau de la moyenne)

In [ ]:
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score)

mae  = mean_absolute_error(yr_te, yr_pred)
mse  = mean_squared_error(yr_te, yr_pred)
rmse = np.sqrt(mse)
r2   = r2_score(yr_te, yr_pred)

print(f'MAE  : {mae:.3f}')
print(f'MSE  : {mse:.3f}   (unité au carré, non interprétable directement)')
print(f'RMSE : {rmse:.3f}')
print(f'R²   : {r2:.3f}   → le modèle explique {r2:.0%} de la variance')
print()
print(f'Écart RMSE - MAE = {rmse - mae:.3f}  → révèle quelques grosses erreurs (outliers)')

## 4. RMSLE — erreur logarithmique (relative)

À utiliser quand la cible est **positive** et s'étale sur plusieurs ordres de grandeur.
Elle mesure l'erreur **relative** et pénalise davantage la **sous-estimation**.

In [ ]:
from sklearn.metrics import mean_squared_log_error

# La cible California Housing est positive → RMSLE calculable
rmsle = np.sqrt(mean_squared_log_error(yr_te, np.clip(yr_pred, 0, None)))
print(f'RMSLE : {rmsle:.4f}')

# 🧩 Complément : MAPE (erreur en %)
from sklearn.metrics import mean_absolute_percentage_error
print(f'MAPE  : {mean_absolute_percentage_error(yr_te, yr_pred):.2%}')

### 📊 Visualiser : valeurs prédites vs réelles

Un bon modèle aligne les points sur la diagonale. Les résidus montrent où il se trompe.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].scatter(yr_te, yr_pred, alpha=0.3, s=10)
lims = [yr_te.min(), yr_te.max()]
axes[0].plot(lims, lims, '--', color='red', label='prédiction parfaite')
axes[0].set_xlabel('Valeur réelle'); axes[0].set_ylabel('Valeur prédite')
axes[0].set_title(f'Prédit vs Réel (R² = {r2:.3f})'); axes[0].legend()

residus = yr_te - yr_pred
axes[1].scatter(yr_pred, residus, alpha=0.3, s=10)
axes[1].axhline(0, color='red', ls='--')
axes[1].set_xlabel('Valeur prédite'); axes[1].set_ylabel('Résidu (réel - prédit)')
axes[1].set_title('Résidus')
plt.tight_layout(); plt.show()

---
# PARTIE III — Métriques de Clustering

On génère des groupes synthétiques avec `make_blobs`. En clustering il n'y a **pas de labels** :
on juge la **géométrie** (cohésion + séparation). On garde ici les vrais groupes uniquement pour l'**ARI**.

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

Xc, y_vrai = make_blobs(n_samples=600, centers=4, cluster_std=1.0, random_state=42)

km = KMeans(n_clusters=4, random_state=42, n_init=10).fit(Xc)
labels = km.labels_

plt.figure(figsize=(6, 5))
plt.scatter(Xc[:, 0], Xc[:, 1], c=labels, cmap='viridis', s=15)
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
            c='red', marker='X', s=200, label='centroïdes')
plt.title('K-Means (K=4)'); plt.legend(); plt.show()

## 1 à 3. Silhouette, Davies-Bouldin, Inertie

- **Silhouette** ∈ [-1, 1] : **haut = mieux** (cohésion + séparation)
- **Davies-Bouldin** : **bas = mieux** (0 idéal) ⚠️ sens inversé
- **Inertie** : somme des distances² aux centroïdes (propre à K-Means)

In [ ]:
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             calinski_harabasz_score)

print(f'Silhouette        : {silhouette_score(Xc, labels):.3f}   (haut = mieux)')
print(f'Davies-Bouldin    : {davies_bouldin_score(Xc, labels):.3f}   (BAS = mieux)')
print(f'Calinski-Harabasz : {calinski_harabasz_score(Xc, labels):.1f}   (haut = mieux)')
print(f'Inertie (K-Means) : {km.inertia_:.1f}')

### 📐 Méthode du coude + Silhouette pour choisir K

L'inertie baisse toujours avec K : on cherche le **coude**. On croise avec la silhouette (qui, elle, a un maximum).

In [ ]:
Ks = range(2, 9)
inerties, silhouettes = [], []
for k in Ks:
    m = KMeans(n_clusters=k, random_state=42, n_init=10).fit(Xc)
    inerties.append(m.inertia_)
    silhouettes.append(silhouette_score(Xc, m.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(list(Ks), inerties, 'o-'); axes[0].set_title('Méthode du coude')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertie')
axes[1].plot(list(Ks), silhouettes, 'o-', color='green'); axes[1].set_title('Silhouette selon K')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette')
plt.tight_layout(); plt.show()

meilleur_k = list(Ks)[int(np.argmax(silhouettes))]
print(f'K optimal selon la silhouette : {meilleur_k}')

## 4. Adjusted Rand Index (ARI) — métrique externe

Quand on **connaît** les vrais groupes (validation/benchmark), l'ARI compare le clustering à la vérité.
1.0 = identique, ~0 = aléatoire. Insensible au **nom** des clusters.

In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, v_measure_score

print(f'ARI       : {adjusted_rand_score(y_vrai, labels):.3f}')
print(f'NMI       : {normalized_mutual_info_score(y_vrai, labels):.3f}')
print(f'V-measure : {v_measure_score(y_vrai, labels):.3f}')

print('\n⚠️ L\'inertie n\'a PAS de sens pour DBSCAN (pas de centroïdes) — voir le cours §24.')

---
# ✅ Synthèse

| Famille | Métriques clés | Réflexe |
|---------|----------------|---------|
| **Classification** | Accuracy, Precision, Recall, F1, AUC | Matrice de confusion d'abord ; déséquilibré → F1/Recall/AUC |
| **Régression** | MAE, MSE, RMSE, RMSLE, R² | Rapporter **RMSE + R²** ; MAE si outliers ; RMSLE si cible > 0 dispersée |
| **Clustering** | Silhouette, Davies-Bouldin, Inertie, ARI | Interne (géométrie) ; ARI si labels ; inertie = K-Means |

> 🔑 **Règle d'or** : la métrique suit le **coût métier** des erreurs, jamais l'inverse.

### 🛠️ À vous de jouer
Reprenez ce notebook avec **vos** données de TP (`clients_telecom.csv`, `loyers_abidjan.csv`, `demandes_credit.csv`)
et calculez les métriques adaptées à chaque problème.

---
*📘 Module Data Science — Les Métriques d'Évaluation | Bootcamp Data Science*